In [2]:
pip install bitsandbytes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_PATH = r"D:\university\Resesarch\RD\models\Qwen2-VL-2B-Instruct"
ADAPTER_PATH = r"C:\Users\hp\Downloads\Downloads\c3_lora_run3_weights"
IMAGE_PATH = r"D:\university\Resesarch\dataset\bugs\bug.4752.jpg"

PROMPT_TEXT = "Detect defects in this image."

# 1. 4-bit Quantization Config (matches your Kaggle training setup)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading processor and 4-bit base model...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_PATH)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0}  # Pin everything to GPU 0 to prevent meta-device offloading
)

# 2. Attach the LoRA adapter
print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

# 3. Prepare Image & Prompt
image = Image.open(IMAGE_PATH).convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT_TEXT}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

# 4. Generate prediction
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=128)

trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
prediction = processor.batch_decode(trimmed_ids, skip_special_tokens=True)[0].strip()

print("\n" + "=" * 50)
print("FINE-TUNED MODEL PREDICTION:")
print(prediction)
print("=" * 50)

Loading processor and 4-bit base model...


Loading weights: 100%|██████████| 729/729 [00:11<00:00, 61.21it/s] 


Attaching LoRA adapter...

FINE-TUNED MODEL PREDICTION:
In this image, the loading bar is missing, which is causing the app to display a blank page. Additionally, the loading bar icon is missing, which is also causing the app to display a blank page.


In [5]:
import os
import re
import json
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_PATH = r"D:\university\Resesarch\RD\models\Qwen2-VL-2B-Instruct"
ADAPTER_PATH = r"C:\Users\hp\Downloads\Downloads\c3_lora_run3_weights"
TEST_JSON_PATH = r"D:\university\Resesarch\dataset\test.json"
OUTPUT_METRICS_PATH = r"D:\university\Resesarch\eval_metrics_run3.json"

# 1. 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading processor and 4-bit base model...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_PATH)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0}
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

# 2. Load ground truth
with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_samples = json.load(f)

print(f"Loaded {len(test_samples)} test samples.")

def parse_json_output(text):
    """Robust extractor for JSON payload."""
    try:
        # Match outermost json structure
        match = re.search(r"\{.*?\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return None

y_true_defect = []
y_pred_defect = []

y_true_type = []
y_pred_type = []

# Optional: slice to smaller count (e.g., test_samples[:100]) for quick verification
eval_subset = test_samples

for sample in tqdm(eval_subset, desc="Evaluating"):
    raw_img_path = sample["images"][0]
    
    # Resolve image path relative to dataset folder if local-vision-oracle root differs
    img_path = raw_img_path.replace("D:/university/Resesarch/local-vision-oracle/dataset/", "D:/university/Resesarch/dataset/")
    img_path = os.path.normpath(img_path)
    
    if not os.path.exists(img_path):
        continue

    # Extract ground truth
    gt_content = json.loads(sample["messages"][1]["content"])
    gt_has_defect = bool(gt_content.get("has_defect", False))
    gt_defect_type = gt_content.get("defect_type", "clean" if not gt_has_defect else "unknown")

    # Prompt user message
    user_prompt = sample["messages"][0]["content"].replace("<image>", "").strip()
    
    image = Image.open(img_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128)

    trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    prediction_raw = processor.batch_decode(trimmed_ids, skip_special_tokens=True)[0].strip()

    pred_parsed = parse_json_output(prediction_raw)

    if pred_parsed and isinstance(pred_parsed, dict):
        pred_has_defect = bool(pred_parsed.get("has_defect", False))
        pred_defect_type = pred_parsed.get("defect_type", "clean" if not pred_has_defect else "unknown")
    else:
        # Fallback keyword match if JSON parsing fails
        pred_has_defect = "true" in prediction_raw.lower() or "defect" in prediction_raw.lower()
        pred_defect_type = "unknown" if pred_has_defect else "clean"

    y_true_defect.append(gt_has_defect)
    y_pred_defect.append(pred_has_defect)

    y_true_type.append(gt_defect_type)
    y_pred_type.append(pred_defect_type)

# 3. Calculate metrics
binary_metrics = {
    "accuracy": accuracy_score(y_true_defect, y_pred_defect),
    "f1_binary": f1_score(y_true_defect, y_pred_defect, zero_division=0),
    "precision_binary": precision_score(y_true_defect, y_pred_defect, zero_division=0),
    "recall_binary": recall_score(y_true_defect, y_pred_defect, zero_division=0),
}

multiclass_metrics = {
    "macro_f1": f1_score(y_true_type, y_pred_type, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_true_type, y_pred_type, average="weighted", zero_division=0),
    "classification_report": classification_report(y_true_type, y_pred_type, zero_division=0, output_dict=True)
}

final_results = {
    "total_evaluated": len(y_true_defect),
    "binary_defect_detection": binary_metrics,
    "multiclass_defect_classification": multiclass_metrics
}

print("\n" + "=" * 50)
print("--- BINARY DETECTION (has_defect) ---")
for k, v in binary_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n--- MULTI-CLASS CLASSIFICATION REPORT ---")
print(classification_report(y_true_type, y_pred_type, zero_division=0))
print("=" * 50)

with open(OUTPUT_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2)

print(f"\nSaved metrics to: {OUTPUT_METRICS_PATH}")

Loading processor and 4-bit base model...


Loading weights: 100%|██████████| 729/729 [00:23<00:00, 30.76it/s]


Attaching LoRA adapter...
Loaded 600 test samples.


Evaluating:   2%|▏         | 9/600 [19:02<20:50:20, 126.94s/it]


KeyboardInterrupt: 